In [55]:
import numpy as np
import pandas as pd

In [56]:
train = pd.read_excel('../data/training.xlsx')
train.head(5)

,user_id,video_id,name_en,description_en,Rank
0,98234,13,Alive!,"In this compelling documentary, five HIV-posit...",1
1,19357,13,Alive!,"In this compelling documentary, five HIV-posit...",1
2,151726,13,Alive!,"In this compelling documentary, five HIV-posit...",1
3,69657,13,Alive!,"In this compelling documentary, five HIV-posit...",1
4,7935,13,Alive!,"In this compelling documentary, five HIV-posit...",1


In [57]:
test = pd.read_excel("../data/test-Ans.xlsx")
test.head(5)

,user_id,video_id,name_en,description_en,Rank
0,191776,13,Alive!,"In this compelling documentary, five HIV-posit...",5
1,144905,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...,1
2,149531,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...,5
3,69657,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...,5
4,201881,14,Appropriate Behavior,Shirin is struggling to become an ideal Persia...,2


In [58]:
train = train.drop(columns=['name_en','description_en'])
test = test.drop(columns=['name_en','description_en'])
train.head(5)

,user_id,video_id,Rank
0,98234,13,1
1,19357,13,1
2,151726,13,1
3,69657,13,1
4,7935,13,1


In [59]:
vid_avg_rank = train[['video_id','Rank']].groupby('video_id').mean()
vid_avg_rank

,Rank
video_id,
13,1.083333
14,2.876712
15,2.247368
16,3.066667
17,3.090909
...,...
1242,3.161616
1243,3.904930
1244,3.246575


In [60]:
uid_avg_rank = train[['user_id','Rank']].groupby('user_id').mean()
uid_avg_rank

,Rank
user_id,
1,1.517915
5,5.000000
7,5.000000
11,3.000000
15,3.645833
...,...
224050,1.000000
224051,5.000000
224055,4.000000


In [61]:
vid_avg_rank = vid_avg_rank.reset_index()
vid_avg_rank['VRank'] = vid_avg_rank['Rank']
vid_avg_rank.pop('Rank')
vid_avg_rank.head(5)

,video_id,VRank
0,13,1.083333
1,14,2.876712
2,15,2.247368
3,16,3.066667
4,17,3.090909


In [62]:
uid_avg_rank = uid_avg_rank.reset_index()
uid_avg_rank['URank'] = uid_avg_rank['Rank']
uid_avg_rank.pop('Rank')
uid_avg_rank.head(5)

,user_id,URank
0,1,1.517915
1,5,5.000000
2,7,5.000000
3,11,3.000000
4,15,3.645833


In [63]:
test = pd.merge(test, vid_avg_rank, how='left')
test = pd.merge(test, uid_avg_rank, how='left')
test.head(5)

,user_id,video_id,Rank,VRank,URank
0,191776,13,5,1.083333,4.077670
1,144905,14,1,2.876712,2.833333
2,149531,14,5,2.876712,4.200000
3,69657,14,5,2.876712,4.063218
4,201881,14,2,2.876712,4.102564


In [64]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 94089 entries, 0 to 94088
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   user_id   94089 non-null  int64  
 1   video_id  94089 non-null  int64  
 2   Rank      94089 non-null  int64  
 3   VRank     94089 non-null  float64
 4   URank     86128 non-null  float64
dtypes: float64(2), int64(3)
memory usage: 4.3 MB


In [65]:
uid_avg_rank.mean()

user_id    133096.631453
URank           2.791864
dtype: float64

In [66]:
test.URank.fillna(2.791864, inplace=True)
#test.URank.fillna(2.5, inplace=True)
#test.URank.fillna(1, inplace=True)
#test.URank.fillna(5, inplace=True)

In [67]:
from math import sqrt

test['Predicted_Rank'] = 0.0

for i in range(0, test.shape[0]):
    test['Predicted_Rank'][i] = round(sqrt(test['VRank'][i] * test['URank'][i])) 
    
test.head(5)

C:/Users/user\AppData\Local\Temp\ipykernel_7660\2343667048.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test['Predicted_Rank'][i] = round(sqrt(test['VRank'][i] * test['URank'][i]))


,user_id,video_id,Rank,VRank,URank,Predicted_Rank
0,191776,13,5,1.083333,4.077670,2.0
1,144905,14,1,2.876712,2.833333,3.0
2,149531,14,5,2.876712,4.200000,3.0
3,69657,14,5,2.876712,4.063218,3.0
4,201881,14,2,2.876712,4.102564,3.0


In [68]:
mse = np.mean((test.Rank - test.Predicted_Rank) ** 2)
rmse = np.sqrt(mse)
print('RMSE =', rmse)

RMSE = 1.5260285686371193


In [69]:
test.pop('Rank')
test.pop('VRank')
test.pop('URank')

test.to_csv("result.csv", index = False)